# Scan Geo 05 — Análisis de Población Expuesta por Punto de Medición y Radio

Este notebook realiza el análisis espacial de **exposición poblacional en el entorno inmediato de cada sensor acústico** de la ciudad:

- **Población por Manzana Censal:** Cruce con la cartografía censal del INE a nivel de manzana (`Poblacion_Manzanas_ValdiviaOct26.geojson`).
- **Atributo `n_per`:** Representa el número total de personas que habitan en cada manzana delimitada.
- **Intersección Areal y Porcentajes de Población:**
  - Siguiendo la metodología de análisis espacial por radio (similar a los parches de morfología urbana de Liu en el paso de análisis), se calcula la fracción geométrica del área de la manzana que cae dentro del círculo de radio $R$ (buffer en metros).
  - Mediante **interpolación dasimétrica por ponderación de área**, se estima la población aportada por cada manzana al entorno del sensor:
    $$\text{Porcentaje de Manzana en Radio} = \left(\frac{\text{Área de Intersección}}{\text{Área Total de la Manzana}}\right) \times 100$$
    $$\text{Población Estimada en Radio} = n\_per \times \left(\frac{\text{Área de Intersección}}{\text{Área Total de la Manzana}}\right)$$
  - Se calcula el porcentaje que representa cada manzana respecto al total de población en el radio del sensor, así como el porcentaje sobre el total de la ciudad.
- **Métricas Urbanas de Densidad:** Densidad habitacional bruta en habitantes por hectárea ($\text{hab/ha}$) y por kilómetro cuadrado ($\text{hab/km}^2$).
- **Productos Generados:**
  - Tablas resumen por punto de medición y desglose detallado por manzana (`.csv`).
  - Dataset estructurado con metadatos (`.json`).
  - Capa vectorial de polígonos de buffer enriquecida (`.geojson`).
  - Visualizaciones analíticas comparativas (ruido $L_{Aeq}$ vs población) y mapa general (`.png`).
  - Mapa interactivo web con Folium (`.html`).

---

## 1. Importación de Librerías y Variables Maestras de Configuración

**Función de esta sección:**
- Importar las librerías científicas y geoespaciales requeridas (`geopandas`, `pandas`, `shapely`, `matplotlib`, `folium`).
- Definir como variables principales de configuración:
  1. `NOMBRE_CIUDAD`: Etiqueta de la ciudad a procesar (ej. `valdivia_mar2014` o `valdivia`), manteniendo total coherencia con los pasos anteriores.
  2. `RADIO_SCAN_M`: Radio del círculo de análisis alrededor de cada sensor expresado en metros (ej. `150.0`).
- Establecer las rutas relativas de insumos y salidas del flujo de trabajo.

In [1]:
# ============================================================
# 1. Librerías y Configuración General
# ============================================================
import os
import sys
import json
import time
import math
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from shapely.geometry import Point, Polygon, box
import folium
from folium import FeatureGroup, LayerControl

# --- VARIABLES MAESTRAS DE CONFIGURACIÓN ---
NOMBRE_CIUDAD = "valdivia_mar2014"   # Etiqueta de la ciudad (ej: valdivia_mar2014, valdivia)
RADIO_SCAN_M  = 150.0                # Radio de análisis del buffer en metros (ej. 150.0, 300.0)

# Rutas estándar del proyecto
MODELO_01     = Path("01_clas")
INPUT_POB_DIR = Path("05_pob/00_in")
DATOS_OUT     = Path("05_pob")
FIGS_DIR      = DATOS_OUT / "01_fig"

DATOS_OUT.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)

print(f"=== PASO 5: ANÁLISIS DE POBLACIÓN POR SENSOR Y RADIO: {NOMBRE_CIUDAD.upper()} ===")
print(f"Radio de análisis configurado: {RADIO_SCAN_M} metros")
print(f"Carpeta de insumos de población: {INPUT_POB_DIR.resolve()}")
print(f"Carpeta de salidas: {DATOS_OUT.resolve()}")

=== PASO 5: ANÁLISIS DE POBLACIÓN POR SENSOR Y RADIO: VALDIVIA_MAR2014 ===
Radio de análisis configurado: 150.0 metros
Carpeta de insumos de población: /home/oscar/Escritorio/_Final_15Sep26/scan_completo/05_pob/00_in
Carpeta de salidas: /home/oscar/Escritorio/_Final_15Sep26/scan_completo/05_pob


## 2. Carga y Normalización de Puntos de Medición Acústica

**Función de esta sección:**
- Cargar el archivo de mediciones de la ciudad estrictamente según la variable `NOMBRE_CIUDAD`:
  `01_clas/mediciones_aumilab_{NOMBRE_CIUDAD.lower()}.json`.
- Estandarizar y limpiar los nombres de campos (`sensor_id`, `calle_sensor`, `latitud`, `longitud`, `decibels`).
- Deducir y normalizar la jerarquía vial según la categoría OGUC (`Troncal`, `Colectora`, `Servicio`, `Local`, etc.).
- Construir un `GeoDataFrame` en coordenadas geográficas WGS84 (`EPSG:4326`) y proyectarlo a Web Mercator métrico (`EPSG:3857`) para mediciones euclidianas exactas en metros.

In [2]:
# ============================================================
# 2. Cargar y Estandarizar Sensores Acústicos
# ============================================================
def deducir_tipo_campana(sensor_id):
    """Deduce el tipo de vía a partir del prefijo del ID si no está especificado."""
    s = str(sensor_id).upper()
    if s.startswith("T"): return "Troncal"
    elif s.startswith("C"): return "Colectora"
    elif s.startswith("S"): return "Servicio"
    elif s.startswith("L"): return "Local"
    elif s.startswith("E") or s.startswith("EM"): return "Estaciones"
    return "Otro"

def normalizar_tipo_campana(row):
    """Estandariza la categoría OGUC de la vía."""
    cat = row.get("categoria_oguc")
    if pd.notna(cat) and str(cat).strip().lower() not in ["none", "nan", ""]:
        c = str(cat).strip().capitalize()
        if c.startswith("Troncal"): return "Troncal"
        elif c.startswith("Colect"): return "Colectora"
        elif c.startswith("Servic"): return "Servicio"
        elif c.startswith("Local"): return "Local"
        elif c.startswith("Expres"): return "Expresa"
        elif c.startswith("Estacion"): return "Estaciones"
        return c
    return deducir_tipo_campana(row.get("sensor_id", row.get("id", "")))

# Colores estandarizados por jerarquía vial para gráficos y mapas
COLORES_CAMPANA = {
    "Troncal": "#dc2626",      # Rojo
    "Colectora": "#ea580c",    # Naranja
    "Servicio": "#d97706",     # Ámbar
    "Local": "#2563eb",        # Azul
    "Estaciones": "#7c3aed",   # Violeta
    "Otro": "#64748b"          # Gris pizarra
}

# Cargar archivo de mediciones de la ciudad seleccionada
archivo_med_json = MODELO_01 / f"mediciones_aumilab_{NOMBRE_CIUDAD.lower()}.json"
if not archivo_med_json.exists():
    raise FileNotFoundError(f"No se encontró el archivo de mediciones: {archivo_med_json}")

with open(archivo_med_json, "r", encoding="utf-8") as f:
    med_raw = json.load(f)

df_sensors = pd.DataFrame(med_raw)

# Estandarización de nombres de columnas
df_sensors["sensor_id"]    = df_sensors["id"].astype(str)
df_sensors["calle_sensor"] = df_sensors.get("calle", "s/n").fillna("s/n")
df_sensors["latitud"]      = df_sensors["lat"].astype(float)
df_sensors["longitud"]     = df_sensors["lon"].astype(float)
df_sensors["decibels"]     = df_sensors.get("la_eq", df_sensors.get("decibels", 0.0)).astype(float)

# Asignar tipo_campana_medicion con prioridad en categoria_oguc
if "tipo_campana_medicion" not in df_sensors.columns or df_sensors["tipo_campana_medicion"].eq("Otro").all():
    df_sensors["tipo_campana_medicion"] = df_sensors.apply(normalizar_tipo_campana, axis=1)

# Construir GeoDataFrame WGS84 y proyectar a Web Mercator (EPSG:3857)
gdf_sensors_4326 = gpd.GeoDataFrame(
    df_sensors,
    geometry=[Point(xy) for xy in zip(df_sensors["longitud"], df_sensors["latitud"])],
    crs=4326
)
gdf_sensors_3857 = gdf_sensors_4326.to_crs(3857)

print(f"✔ Sensores cargados para [{NOMBRE_CIUDAD}]: {len(gdf_sensors_3857)} puntos desde {archivo_med_json.name}")
print(f"  Rango de decibeles medidos (LAeq): {df_sensors['decibels'].min():.1f} a {df_sensors['decibels'].max():.1f} dB")
print(f"  Categorías de vías representadas: {df_sensors['tipo_campana_medicion'].value_counts().to_dict()}")

✔ Sensores cargados para [valdivia_mar2014]: 100 puntos desde mediciones_aumilab_valdivia_mar2014.json
  Rango de decibeles medidos (LAeq): 49.6 a 75.1 dB
  Categorías de vías representadas: {'Local': 55, 'Servicio': 31, 'Colectora': 14}


## 3. Carga y Procesamiento de la Cartografía Censal de Población (Manzanas)

**Función de esta sección:**
- Localizar el archivo GeoJSON de población dentro de `05_pob/00_in/`. Se incluye una función inteligente de detección que busca archivos coincidentes con el nombre de la ciudad o el archivo censal de Valdivia (`Poblacion_Manzanas_ValdiviaOct26.geojson`).
- Validar y convertir el campo clave **`n_per`** (número de personas censadas por manzana) a tipo numérico de punto flotante, asegurando que registros con formato texto o nulos no alteren las sumas.
- Proyectar la cartografía de manzanas a metros (`EPSG:3857`).
- Calcular el área geométrica exacta de cada manzana en metros cuadrados ($m^2$).
- Construir un **índice espacial R-Tree (`sindex`)** para permitir búsquedas geoespaciales de alta velocidad entre los círculos de buffer y las manzanas.
- Calcular la población total comunal para el cálculo de proporciones relativas.

In [3]:
# ============================================================
# 3. Cargar y Procesar Cartografía de Manzanas Censales (05_pob/00_in)
# ============================================================
def detectar_archivo_poblacion(input_dir, ciudad):
    """Localiza el archivo GeoJSON de población correspondiente a la ciudad."""
    # 1. Búsqueda directa por combinaciones de nombre
    candidatos = [
        input_dir / f"poblacion_{ciudad.lower()}.geojson",
        input_dir / f"Poblacion_Manzanas_{ciudad}.geojson",
        input_dir / f"Poblacion_Manzanas_{ciudad.capitalize()}.geojson",
    ]
    for c in candidatos:
        if c.exists():
            return c
    
    # 2. Búsqueda por la raíz del nombre (ej: 'valdivia' en 'valdivia_mar2014')
    raiz = ciudad.split("_")[0].lower()
    coincidencias = [f for f in input_dir.glob("*.geojson") if raiz in f.name.lower()]
    if coincidencias:
        return coincidencias[0]
        
    # 3. Si existe un único archivo geojson en 05_pob/00_in, seleccionarlo
    todos = list(input_dir.glob("*.geojson"))
    if len(todos) == 1:
        return todos[0]
        
    raise FileNotFoundError(f"No se encontró archivo GeoJSON de población para '{ciudad}' en {input_dir}")

archivo_pob_geojson = detectar_archivo_poblacion(INPUT_POB_DIR, NOMBRE_CIUDAD)
print(f"Cargando cartografía censal desde: {archivo_pob_geojson.name}...")

gdf_pob_4326 = gpd.read_file(archivo_pob_geojson)

# Verificación y conversión del campo 'n_per' a numérico
if "n_per" not in gdf_pob_4326.columns:
    raise KeyError(f"El archivo {archivo_pob_geojson.name} no contiene la columna obligatoria 'n_per'.")

gdf_pob_4326["n_per"] = pd.to_numeric(gdf_pob_4326["n_per"], errors="coerce").fillna(0.0)

# Proyección a Web Mercator métrico (EPSG:3857) y cálculo de área
gdf_pob_3857 = gdf_pob_4326.to_crs(3857)
gdf_pob_3857["area_manzana_m2"] = gdf_pob_3857.geometry.area

# Identificador de manzana normalizado
if "MANZENT" in gdf_pob_3857.columns:
    gdf_pob_3857["id_manzana"] = gdf_pob_3857["MANZENT"].astype(str)
elif "COD_MANZANA" in gdf_pob_3857.columns:
    gdf_pob_3857["id_manzana"] = gdf_pob_3857["COD_MANZANA"].astype(str)
else:
    gdf_pob_3857["id_manzana"] = [f"MZ_{i+1}" for i in range(len(gdf_pob_3857))]

# Construcción de índice espacial R-Tree
sindex_pob = gdf_pob_3857.sindex

poblacion_total_ciudad = float(gdf_pob_3857["n_per"].sum())
total_manzanas = len(gdf_pob_3857)

print(f"✔ Cartografía de manzanas cargada exitosamente:")
print(f"  Total de manzanas censales: {total_manzanas:,}")
print(f"  Población total registrada en la cartografía: {poblacion_total_ciudad:,.0f} personas")
print(f"  Promedio de habitantes por manzana: {gdf_pob_3857['n_per'].mean():.1f} (máx: {gdf_pob_3857['n_per'].max():.0f})")
print(f"  Índice espacial R-Tree generado para consultas optimizadas.")

Cargando cartografía censal desde: Poblacion_Manzanas_ValdiviaOct26.geojson...


ERROR 1: PROJ: proj_create_from_database: Open of /home/oscar/miniconda3/envs/geo03-gpu/share/proj failed


✔ Cartografía de manzanas cargada exitosamente:
  Total de manzanas censales: 1,962
  Población total registrada en la cartografía: 125,743 personas
  Promedio de habitantes por manzana: 64.1 (máx: 983)
  Índice espacial R-Tree generado para consultas optimizadas.


## 4. Definición de la Función de Escaneo Espacial por Sensor

**Función de esta sección:**
- Implementar la función `ejecutar_scan_poblacion_sensor(sensor_row, radio_m, ...)` que realiza el análisis geométrico y demográfico para un sensor específico:
  1. Genera el buffer circular de radio $R$ (en metros) centrado en las coordenadas del sensor.
  2. Consulta el índice espacial R-Tree para obtener únicamente las manzanas que intersectan el círculo en tiempo constante.
  3. Realiza la intersección geométrica exacta de polígonos:
     $$\text{Área Intersección} = \text{Área}(\text{Manzana} \cap \text{Círculo})$$
  4. Calcula el **porcentaje de la manzana dentro del radio**:
     $$\%\text{ Manzana en Radio} = \left(\frac{\text{Área Intersección}}{\text{Área Total Manzana}}\right) \times 100$$
  5. Estima la **población dentro del radio** proveniente de esa manzana:
     $$\text{Población Estimada} = n\_per \times \left(\frac{\text{Área Intersección}}{\text{Área Total Manzana}}\right)$$
  6. Para cada manzana calcula el **porcentaje de aporte** que representa sobre la población total que rodea al sensor:
     $$\%\text{ Aporte al Sensor} = \left(\frac{\text{Población Estimada}}{\text{Población Total en Radio}}\right) \times 100$$
  7. Calcula métricas urbanas globales del sensor: densidad en habitantes por hectárea ($\text{hab/ha}$), densidad en $\text{hab/km}^2$, porcentaje respecto al total de la ciudad y cobertura de manzanas urbanas dentro del círculo.

In [4]:
# ============================================================
# 4. Función de Escaneo Espacial de Población por Sensor
# ============================================================
def ejecutar_scan_poblacion_sensor(sensor_row, radio_m, gdf_manzanas, sindex_mz, pob_ciudad_total):
    """
    Calcula la intersección geométrica y la población estimada dentro
    del radio de análisis para un punto de medición.
    """
    sensor_id = str(sensor_row["sensor_id"])
    calle = str(sensor_row.get("calle_sensor", "s/n"))
    campana = str(sensor_row.get("tipo_campana_medicion", "Otro"))
    decibels = float(sensor_row.get("decibels", 0.0))
    pt_geom = sensor_row.geometry
    
    # 1. Crear buffer circular del sensor
    circulo = pt_geom.buffer(radio_m, resolution=128)
    area_buffer_m2 = circulo.area
    area_buffer_ha = area_buffer_m2 / 10000.0  # 1 hectárea = 10,000 m²
    
    # 2. Filtrar manzanas candidatas mediante el índice espacial
    idxs_candidatas = list(sindex_mz.query(circulo, predicate="intersects"))
    
    manzanas_detalle = []
    pob_total_en_radio = 0.0
    area_urbana_inter_total_m2 = 0.0
    
    # 3. Intersección geométrica con cada manzana candidata
    for i_mz in idxs_candidatas:
        mz = gdf_manzanas.iloc[i_mz]
        area_mz_total = mz["area_manzana_m2"]
        if area_mz_total <= 0:
            continue
            
        inter = mz.geometry.intersection(circulo)
        if inter.area > 0:
            area_inter_m2 = inter.area
            # Porcentaje del área de la manzana contenida en el radio
            pct_mz_en_radio = (area_inter_m2 / area_mz_total) * 100.0
            
            # Estimación dasimétrica de personas en el radio
            pob_estimada = float(mz["n_per"]) * (area_inter_m2 / area_mz_total)
            
            pob_total_en_radio += pob_estimada
            area_urbana_inter_total_m2 += area_inter_m2
            
            manzanas_detalle.append({
                "sensor_id": sensor_id,
                "calle_sensor": calle,
                "tipo_campana": campana,
                "id_manzana": mz["id_manzana"],
                "n_per_manzana": float(mz["n_per"]),
                "area_manzana_m2": round(area_mz_total, 2),
                "area_interseccion_m2": round(area_inter_m2, 2),
                "porcentaje_manzana_en_radio": round(pct_mz_en_radio, 4),
                "poblacion_estimada_en_radio": round(pob_estimada, 4)
            })
    
    # 4. Calcular porcentaje de aporte de cada manzana respecto al total del círculo
    for item in manzanas_detalle:
        if pob_total_en_radio > 0:
            item["porcentaje_aporte_al_sensor"] = round(
                (item["poblacion_estimada_en_radio"] / pob_total_en_radio) * 100.0, 4
            )
        else:
            item["porcentaje_aporte_al_sensor"] = 0.0

    # 5. Métricas agregadas del punto de medición
    pct_poblacion_ciudad = (pob_total_en_radio / pob_ciudad_total * 100.0) if pob_ciudad_total > 0 else 0.0
    densidad_hab_ha = pob_total_en_radio / area_buffer_ha if area_buffer_ha > 0 else 0.0
    densidad_hab_km2 = pob_total_en_radio / (area_buffer_m2 / 1e6) if area_buffer_m2 > 0 else 0.0
    pct_cobertura_urbana = (area_urbana_inter_total_m2 / area_buffer_m2 * 100.0) if area_buffer_m2 > 0 else 0.0
    
    sensor_resumen = {
        "sensor_id": sensor_id,
        "calle_sensor": calle,
        "tipo_campana": campana,
        "decibels": decibels,
        "latitud": float(sensor_row["latitud"]),
        "longitud": float(sensor_row["longitud"]),
        "scan_radius_m": float(radio_m),
        "area_buffer_m2": round(area_buffer_m2, 2),
        "area_buffer_ha": round(area_buffer_ha, 4),
        "cantidad_manzanas_en_radio": len(manzanas_detalle),
        "poblacion_total_en_radio": round(pob_total_en_radio, 2),
        "porcentaje_poblacion_ciudad": round(pct_poblacion_ciudad, 6),
        "densidad_hab_ha": round(densidad_hab_ha, 2),
        "densidad_hab_km2": round(densidad_hab_km2, 2),
        "area_manzanas_en_radio_m2": round(area_urbana_inter_total_m2, 2),
        "porcentaje_cobertura_manzanas": round(pct_cobertura_urbana, 2)
    }
    
    return sensor_resumen, manzanas_detalle, circulo

## 5. Ejecución del Análisis Espacial para Todos los Puntos de Medición

**Función de esta sección:**
- Iterar secuencialmente sobre la totalidad de sensores acústicos de la ciudad.
- Ejecutar la función de escaneo para cada punto con el radio `RADIO_SCAN_M`.
- Consolidar dos estructuras tabulares principales:
  1. `df_sensores_resumen`: Una fila por sensor con sus métricas demográficas consolidadas.
  2. `df_manzanas_detalle`: Una fila por cada par (sensor, manzana intersectada) con las métricas de área, porcentaje y población.
- Construir un `GeoDataFrame` con las geometrías poligonales del buffer circular (`gdf_buffers_pob`).
- Imprimir un resumen estadístico detallado en consola.

In [5]:
# ============================================================
# 5. Ejecución del Escaneo Masivo
# ============================================================
print(f"Ejecutando escaneo espacial de población con radio={RADIO_SCAN_M}m para {len(gdf_sensors_3857)} sensores...")
t_inicio = time.time()

lista_resumen_sensores = []
lista_detalle_manzanas = []
geometrias_buffer = []

for i, (idx, s_row) in enumerate(gdf_sensors_3857.iterrows(), start=1):
    res_s, det_mz, circ_geom = ejecutar_scan_poblacion_sensor(
        sensor_row=s_row,
        radio_m=RADIO_SCAN_M,
        gdf_manzanas=gdf_pob_3857,
        sindex_mz=sindex_pob,
        pob_ciudad_total=poblacion_total_ciudad
    )
    lista_resumen_sensores.append(res_s)
    lista_detalle_manzanas.extend(det_mz)
    geometrias_buffer.append(circ_geom)
    
    if i % 25 == 0 or i == len(gdf_sensors_3857):
        print(f"  [{i}/{len(gdf_sensors_3857)}] Sensores procesados ({time.time() - t_inicio:.2f} s)...")

# Construir DataFrames
df_sensores_resumen = pd.DataFrame(lista_resumen_sensores)
df_manzanas_detalle = pd.DataFrame(lista_detalle_manzanas)

# GeoDataFrame con geometrías de buffer en EPSG:3857 y EPSG:4326
gdf_buffers_3857 = gpd.GeoDataFrame(df_sensores_resumen, geometry=geometrias_buffer, crs=3857)
gdf_buffers_4326 = gdf_buffers_3857.to_crs(4326)

tiempo_total = time.time() - t_inicio
sensores_con_pob = (df_sensores_resumen["poblacion_total_en_radio"] > 0).sum()

print(f"\n✔ Escaneo completado en {tiempo_total:.2f} segundos:")
print(f"  Total sensores analizados: {len(df_sensores_resumen)}")
print(f"  Sensores con población en radio: {sensores_con_pob} / {len(df_sensores_resumen)} ({sensores_con_pob/len(df_sensores_resumen)*100:.1f}%)")
print(f"  Total intersecciones sensor-manzana: {len(df_manzanas_detalle):,}")
print(f"  Población estimada en radio:")
print(f"    - Mínimo:  {df_sensores_resumen['poblacion_total_en_radio'].min():.1f} hab")
print(f"    - Mediana: {df_sensores_resumen['poblacion_total_en_radio'].median():.1f} hab")
print(f"    - Promedio:{df_sensores_resumen['poblacion_total_en_radio'].mean():.1f} hab")
print(f"    - Máximo:  {df_sensores_resumen['poblacion_total_en_radio'].max():.1f} hab")
print(f"    - Densidad máx: {df_sensores_resumen['densidad_hab_ha'].max():.1f} hab/ha")

Ejecutando escaneo espacial de población con radio=150.0m para 100 sensores...
  [25/100] Sensores procesados (0.14 s)...


  [50/100] Sensores procesados (0.24 s)...
  [75/100] Sensores procesados (0.32 s)...
  [100/100] Sensores procesados (0.43 s)...

✔ Escaneo completado en 0.44 segundos:
  Total sensores analizados: 100
  Sensores con población en radio: 98 / 100 (98.0%)
  Total intersecciones sensor-manzana: 929
  Población estimada en radio:
    - Mínimo:  0.0 hab
    - Mediana: 277.4 hab
    - Promedio:294.1 hab
    - Máximo:  842.3 hab
    - Densidad máx: 119.2 hab/ha


## 6. Exportación de Resultados (CSV, JSON y GeoJSON)

**Función de esta sección:**
- Guardar los resultados en la carpeta `05_pob/` con nombres normalizados según la ciudad y el radio analizado:
  1. `p5_puntos_poblacion_{NOMBRE_CIUDAD.lower()}_r{radio}.csv`: Resumen por sensor acústico.
  2. `p5_detalle_manzanas_{NOMBRE_CIUDAD.lower()}_r{radio}.csv`: Detalle de cada manzana intersectada.
  3. `p5_analisis_poblacion_{NOMBRE_CIUDAD.lower()}_r{radio}.json`: Archivo JSON jerárquico estructurado con metadatos completos del estudio.
  4. `p5_buffers_poblacion_{NOMBRE_CIUDAD.lower()}_r{radio}.geojson`: Capa vectorial con los polígonos de los buffers circulares en WGS84 para visualización en SIG (QGIS, ArcGIS o GeoPandas).

In [6]:
# ============================================================
# 6. Exportación de Tablas, GeoJSON y JSON Estructurado
# ============================================================
r_suffix = f"r{int(RADIO_SCAN_M)}"
ciudad_tag = NOMBRE_CIUDAD.lower()

# Rutas de salida
csv_resumen_path = DATOS_OUT / f"p5_puntos_poblacion_{ciudad_tag}_{r_suffix}.csv"
csv_detalle_path = DATOS_OUT / f"p5_detalle_manzanas_{ciudad_tag}_{r_suffix}.csv"
json_salida_path = DATOS_OUT / f"p5_analisis_poblacion_{ciudad_tag}_{r_suffix}.json"
geojson_buf_path = DATOS_OUT / f"p5_buffers_poblacion_{ciudad_tag}_{r_suffix}.geojson"

# 1. Exportar CSV Resumen
df_sensores_resumen.to_csv(csv_resumen_path, index=False, encoding="utf-8")
print(f"✔ CSV resumen guardado: {csv_resumen_path.name} ({len(df_sensores_resumen)} filas)")

# 2. Exportar CSV Detalle de Manzanas
df_manzanas_detalle.to_csv(csv_detalle_path, index=False, encoding="utf-8")
print(f"✔ CSV detalle guardado: {csv_detalle_path.name} ({len(df_manzanas_detalle)} filas)")

# 3. Exportar GeoJSON de Buffers
gdf_buffers_4326.to_file(geojson_buf_path, driver="GeoJSON")
print(f"✔ GeoJSON buffers guardado: {geojson_buf_path.name}")

# 4. Exportar JSON jerárquico completo
estructura_json = {
    "metadata": {
        "ciudad": NOMBRE_CIUDAD,
        "radio_analisis_metros": RADIO_SCAN_M,
        "fecha_procesamiento": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "total_sensores": len(df_sensores_resumen),
        "total_manzanas_ciudad": total_manzanas,
        "poblacion_total_ciudad": poblacion_total_ciudad,
        "archivo_mediciones": str(archivo_med_json.name),
        "archivo_poblacion": str(archivo_pob_geojson.name)
    },
    "estadisticas_generales": {
        "poblacion_radio_promedio": round(df_sensores_resumen["poblacion_total_en_radio"].mean(), 2),
        "poblacion_radio_mediana": round(df_sensores_resumen["poblacion_total_en_radio"].median(), 2),
        "poblacion_radio_maxima": round(df_sensores_resumen["poblacion_total_en_radio"].max(), 2),
        "densidad_promedio_hab_ha": round(df_sensores_resumen["densidad_hab_ha"].mean(), 2),
        "densidad_maxima_hab_ha": round(df_sensores_resumen["densidad_hab_ha"].max(), 2)
    },
    "sensores": []
}

for s_dict in lista_resumen_sensores:
    s_id = s_dict["sensor_id"]
    mzs_s = [m for m in lista_detalle_manzanas if m["sensor_id"] == s_id]
    s_entry = {
        **s_dict,
        "manzanas_intersectadas": mzs_s
    }
    estructura_json["sensores"].append(s_entry)

with open(json_salida_path, "w", encoding="utf-8") as f:
    json.dump(estructura_json, f, indent=2, ensure_ascii=False)

print(f"✔ JSON estructurado guardado: {json_salida_path.name}")

✔ CSV resumen guardado: p5_puntos_poblacion_valdivia_mar2014_r150.csv (100 filas)
✔ CSV detalle guardado: p5_detalle_manzanas_valdivia_mar2014_r150.csv (929 filas)
✔ GeoJSON buffers guardado: p5_buffers_poblacion_valdivia_mar2014_r150.geojson
✔ JSON estructurado guardado: p5_analisis_poblacion_valdivia_mar2014_r150.json


## 7. Visualización y Análisis Gráfico Estático (Matplotlib)

**Función de esta sección:**
- Generar dos figuras gráficas de alta calidad para la memoria técnica y reportes:
  1. `figura_analisis_poblacion_{ciudad}_{radio}.png`:
     - **Subplot Izquierdo:** Relación bivariada entre el Nivel de Ruido Continuo Equivalente ($L_{Aeq}$) y la Población en el radio, coloreada según la categoría de vía OGUC. Permite identificar de inmediato puntos críticos (alto ruido y alta población expuesta).
     - **Subplot Derecho:** Ranking con el Top 15 de sensores que albergan la mayor población en su radio de influencia, identificando la calle y el nivel de decibeles.
  2. `mapa_poblacion_manzanas_{ciudad}_{radio}.png`:
     - Mapa temático general de la ciudad: muestra todas las manzanas coloreadas según su población (`n_per`) en escala logarítmica/gradiente, la ubicación de los sensores acústicos y sus círculos de escaneo destacados.

In [7]:
# ============================================================
# 7. Figuras y Mapas Estáticos (Matplotlib)
# ============================================================
fig_analisis_path = DATOS_OUT / f"figura_analisis_poblacion_{ciudad_tag}_{r_suffix}.png"
fig_mapa_path     = DATOS_OUT / f"mapa_poblacion_manzanas_{ciudad_tag}_{r_suffix}.png"

# --- FIGURA 1: Ruido vs Población y Top Sensores Poblados ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), facecolor="#f8fafc")
fig.suptitle(
    f"{NOMBRE_CIUDAD.upper()} — Análisis de Población Expuesta por Punto de Medición (Radio {RADIO_SCAN_M}m)",
    fontsize=14, fontweight="bold", y=0.98, color="#0f172a"
)

# Panel A: Scatter Ruido vs Población
for campana, grp in df_sensores_resumen.groupby("tipo_campana"):
    c = COLORES_CAMPANA.get(campana, "#64748b")
    ax1.scatter(
        grp["decibels"], grp["poblacion_total_en_radio"],
        color=c, edgecolor="black", linewidth=0.7, s=70, alpha=0.85, label=campana
    )

ax1.set_facecolor("#ffffff")
ax1.set_title("Nivel de Ruido (LAeq) vs Población Estimada en Radio", fontsize=11, fontweight="bold", pad=10)
ax1.set_xlabel("Nivel Sonoro Continuo Equivalente LAeq (dB)", fontsize=10, color="#334155")
ax1.set_ylabel(f"Población Estimada dentro de {RADIO_SCAN_M}m (hab)", fontsize=10, color="#334155")
ax1.grid(True, linestyle="--", alpha=0.35, color="#94a3b8")
ax1.legend(title="Vía OGUC", frameon=True, facecolor="#ffffff", edgecolor="#cbd5e1", fontsize=9)

# Panel B: Ranking Top 15 Sensores más Poblados
top15 = df_sensores_resumen.sort_values(by="poblacion_total_en_radio", ascending=True).tail(15)
colores_barras = [COLORES_CAMPANA.get(cat, "#64748b") for cat in top15["tipo_campana"]]

y_pos = np.arange(len(top15))
ax2.barh(y_pos, top15["poblacion_total_en_radio"], color=colores_barras, edgecolor="black", linewidth=0.5, alpha=0.85)
ax2.set_yticks(y_pos)
etiquetas_top = [f"{row['sensor_id']} ({row['calle_sensor'][:22]})" for _, row in top15.iterrows()]
ax2.set_yticklabels(etiquetas_top, fontsize=9)
ax2.set_facecolor("#ffffff")
ax2.set_title(f"Top 15 Puntos de Medición con Mayor Población en Radio", fontsize=11, fontweight="bold", pad=10)
ax2.set_xlabel(f"Población Estimada (hab)", fontsize=10, color="#334155")
ax2.grid(True, linestyle="--", alpha=0.35, color="#94a3b8", axis="x")

# Anotar valores numéricos al final de cada barra
for i, (_, row) in enumerate(top15.iterrows()):
    ax2.text(row["poblacion_total_en_radio"] + 8, i, f"{row['poblacion_total_en_radio']:.0f} hab ({row['decibels']:.1f} dB)",
             va="center", fontsize=8, color="#0f172a", fontweight="bold")

plt.tight_layout()
plt.savefig(fig_analisis_path, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"✔ Gráfica de análisis guardada: {fig_analisis_path.name}")

# --- FIGURA 2: Mapa General de Población y Sensores ---
fig_map, ax_map = plt.subplots(figsize=(12, 11), facecolor="#f8fafc")
ax_map.set_facecolor("#f1f5f9")

# 1. Dibujar manzanas de la ciudad sombreadas por 'n_per'
gdf_pob_3857.plot(
    column="n_per",
    ax=ax_map,
    cmap="YlOrRd",
    legend=True,
    legend_kwds={
        "label": "Población Censal por Manzana (n_per)",
        "orientation": "horizontal",
        "shrink": 0.6,
        "pad": 0.05
    },
    edgecolor="#94a3b8",
    linewidth=0.3,
    alpha=0.75
)

# 2. Dibujar círculos de los buffers
gdf_buffers_3857.plot(
    ax=ax_map,
    facecolor="none",
    edgecolor="#2563eb",
    linewidth=1.2,
    linestyle="--",
    alpha=0.85
)

# 3. Dibujar centroides de los sensores acústicos
gdf_sensors_3857.plot(
    ax=ax_map,
    color="#0f172a",
    edgecolor="#ffffff",
    markersize=35,
    linewidth=1.0,
    zorder=6
)

ax_map.set_title(
    f"{NOMBRE_CIUDAD.upper()} — Distribución de Población Censal y Sensores Acústicos\nRadios de Análisis: {RADIO_SCAN_M} metros",
    fontsize=13, fontweight="bold", pad=12
)
ax_map.set_xlabel("Coordenadas Web Mercator X (m)", fontsize=9, color="#475569")
ax_map.set_ylabel("Coordenadas Web Mercator Y (m)", fontsize=9, color="#475569")
ax_map.grid(True, linestyle="--", alpha=0.3, color="#94a3b8")

plt.tight_layout()
plt.savefig(fig_mapa_path, dpi=200, bbox_inches="tight")
plt.close(fig_map)
print(f"✔ Mapa temático de población guardado: {fig_mapa_path.name}")

✔ Gráfica de análisis guardada: figura_analisis_poblacion_valdivia_mar2014_r150.png


✔ Mapa temático de población guardado: mapa_poblacion_manzanas_valdivia_mar2014_r150.png


## 8. Construcción del Mapa Interactivo Web (Folium)

**Función de esta sección:**
- Crear una aplicación cartográfica web interactiva y autocontenida (`.html`) utilizando Folium.
- Capas interactivas:
  1. **Manzanas Censales (Valdivia):** Polígonos de manzanas con escala coroplética de color según su población (`n_per`) y tooltips dinámicos con información censal.
  2. **Círculos de Escaneo (Radio $R$ m):** Buffers circulares centrados en cada sensor.
  3. **Marcadores de Sensores:** Íconos interactivos con popups estilizados en HTML que muestran:
     - Nombre del Sensor y Calle
     - Nivel de Ruido $L_{Aeq}$ (dB)
     - Categoría de Vía OGUC
     - Población Total Estimada en el radio
     - Densidad Habitacional ($hab/ha$)
     - Cantidad de Manzanas Intersectadas
     - Cobertura Urbana (%)
- Se exporta a `05_pob/mapa_poblacion_{NOMBRE_CIUDAD.lower()}_r{radio}.html` para visualización directa en cualquier navegador web.

In [8]:
# ============================================================
# 8. Mapa Interactivo Web con Folium
# ============================================================
html_mapa_path = DATOS_OUT / f"mapa_poblacion_{ciudad_tag}_{r_suffix}.html"

# Centro del mapa en coordenadas geográficas WGS84
centro_lat = float(df_sensors["latitud"].mean())
centro_lon = float(df_sensors["longitud"].mean())

m = folium.Map(
    location=[centro_lat, centro_lon],
    zoom_start=13,
    tiles="CartoDB positron",
    control_scale=True
)

# Capa 1: Manzanas Censales de Valdivia
fg_manzanas = FeatureGroup(name="Manzanas Censales (Población n_per)", show=True)

# Normalización de color para las manzanas según n_per
pob_max_mz = max(gdf_pob_4326["n_per"].quantile(0.95), 1.0)

def estilo_manzana(feature):
    n_p = float(feature["properties"].get("n_per", 0))
    ratio = min(n_p / pob_max_mz, 1.0)
    # Gradiente de amarillo (#fed7aa) a rojo oscuro (#991b1b)
    if ratio < 0.05: color_fill = "#f1f5f9"
    elif ratio < 0.25: color_fill = "#fed7aa"
    elif ratio < 0.50: color_fill = "#fb923c"
    elif ratio < 0.75: color_fill = "#ea580c"
    else: color_fill = "#991b1b"
    
    return {
        "fillColor": color_fill,
        "color": "#64748b",
        "weight": 0.6,
        "fillOpacity": 0.5 if ratio > 0 else 0.15
    }

# Campos simplificados para el GeoJSON interactivo
columnas_geo = ["id_manzana", "n_per", "geometry"]
gdf_pob_mini = gdf_pob_4326.copy()
gdf_pob_mini["id_manzana"] = gdf_pob_3857["id_manzana"]

folium.GeoJson(
    gdf_pob_mini[columnas_geo],
    style_function=estilo_manzana,
    tooltip=folium.GeoJsonTooltip(
        fields=["id_manzana", "n_per"],
        aliases=["Código Manzana:", "Habitantes (n_per):"],
        localize=True
    )
).add_to(fg_manzanas)

fg_manzanas.add_to(m)

# Capa 2: Círculos y Sensores Acústicos
fg_sensores = FeatureGroup(name=f"Sensores y Entorno ({RADIO_SCAN_M}m)", show=True)

for _, row in df_sensores_resumen.iterrows():
    lat = row["latitud"]
    lon = row["longitud"]
    s_id = row["sensor_id"]
    calle = row["calle_sensor"]
    campana = row["tipo_campana"]
    decibels = row["decibels"]
    pob_est = row["poblacion_total_en_radio"]
    dens_ha = row["densidad_hab_ha"]
    cant_mz = row["cantidad_manzanas_en_radio"]
    cobertura = row["porcentaje_cobertura_manzanas"]
    
    color_c = COLORES_CAMPANA.get(campana, "#2563eb")
    
    # 1. Círculo de buffer geográfico real
    folium.Circle(
        location=[lat, lon],
        radius=RADIO_SCAN_M,
        color=color_c,
        weight=2,
        fill=True,
        fill_color=color_c,
        fill_opacity=0.18
    ).add_to(fg_sensores)
    
    # 2. Popup estilizado HTML
    html_popup = f"""
    <div style="font-family: system-ui, -apple-system, sans-serif; font-size: 12px; min-width: 250px;">
        <div style="background:{color_c}; color:white; padding:6px 10px; border-radius:4px 4px 0 0; font-weight:bold;">
            Sensor {s_id} — {calle}
        </div>
        <div style="padding: 8px 10px; border: 1px solid #cbd5e1; border-top: none; border-radius: 0 0 4px 4px; background: #ffffff;">
            <table style="width:100%; border-collapse:collapse; font-size:12px;">
                <tr><td style="padding:3px 0; color:#64748b;">Nivel de Ruido:</td><td><b>{decibels:.1f} dB</b> (LAeq)</td></tr>
                <tr><td style="padding:3px 0; color:#64748b;">Vía OGUC:</td><td><b>{campana}</b></td></tr>
                <tr style="border-top:1px solid #e2e8f0;"><td style="padding:4px 0; color:#0f172a; font-weight:600;" colspan="2">Población en Radio {RADIO_SCAN_M}m:</td></tr>
                <tr><td style="padding:3px 0; color:#64748b;">Habitantes Estimados:</td><td><b style="color:#b91c1c; font-size:13px;">{pob_est:,.1f} hab</b></td></tr>
                <tr><td style="padding:3px 0; color:#64748b;">Densidad Habitacional:</td><td><b>{dens_ha:.1f} hab/ha</b></td></tr>
                <tr><td style="padding:3px 0; color:#64748b;">Manzanas en Radio:</td><td>{cant_mz} manzanas</td></tr>
                <tr><td style="padding:3px 0; color:#64748b;">Cobertura de Manzanas:</td><td>{cobertura:.1f}% del círculo</td></tr>
            </table>
        </div>
    </div>
    """
    
    # 3. Marcador puntual
    folium.CircleMarker(
        location=[lat, lon],
        radius=5,
        color="#000000",
        weight=1.5,
        fill=True,
        fill_color=color_c,
        fill_opacity=1.0,
        popup=folium.Popup(html_popup, max_width=320),
        tooltip=f"Sensor {s_id}: {pob_est:.0f} hab | {decibels:.1f} dB"
    ).add_to(fg_sensores)

fg_sensores.add_to(m)
LayerControl().add_to(m)

m.save(str(html_mapa_path))
print(f"✔ Mapa interactivo web guardado exitosamente:")
print(f"  Ruta: {html_mapa_path.resolve()}")

✔ Mapa interactivo web guardado exitosamente:
  Ruta: /home/oscar/Escritorio/_Final_15Sep26/scan_completo/05_pob/mapa_poblacion_valdivia_mar2014_r150.html


## 9. Inspección y Ficha Detallada por Punto de Medición

**Función de esta sección:**
- Proveer la función interactiva `inspeccionar_sensor(sensor_id)` que permite inspeccionar cualquier sensor:
  - Extrae su información resumida y la tabla de todas las manzanas censales que intersectan el círculo de análisis.
  - Genera una vista en zoom con Matplotlib mostrando el círculo exacto del sensor, las geometrías de las manzanas, el nombre de la ciudad en el título y las etiquetas con el porcentaje de cada manzana (%Mz), el porcentaje de población aportada (%Pob) y los habitantes estimados (hab).
  - Se ejecuta automáticamente sobre el punto con mayor población expuesta de la ciudad a modo de ejemplo demostrativo.


In [9]:
# ============================================================
# 9. Función de Ficha e Inspección Individual por Sensor
# ============================================================
def inspeccionar_sensor(sensor_id_consulta):
    """
    Muestra la ficha técnica detallada y gráfica de un sensor en particular.
    """
    res_s = df_sensores_resumen[df_sensores_resumen["sensor_id"] == str(sensor_id_consulta)]
    if res_s.empty:
        print(f"Sensor '{sensor_id_consulta}' no encontrado.")
        return
        
    s_row = res_s.iloc[0]
    detalles = df_manzanas_detalle[df_manzanas_detalle["sensor_id"] == str(sensor_id_consulta)]
    
    print(f"============================================================")
    print(f"FICHA TÉCNICA — SENSOR {s_row['sensor_id']} ({s_row['calle_sensor']})")
    print(f"============================================================")
    print(f"• Vía OGUC: {s_row['tipo_campana']} | Nivel de Ruido: {s_row['decibels']:.1f} dB")
    print(f"• Radio de Análisis: {s_row['scan_radius_m']} m | Área del círculo: {s_row['area_buffer_ha']:.2f} ha")
    print(f"• Población Estimada en Radio: {s_row['poblacion_total_en_radio']:.1f} habitantes")
    print(f"• Densidad Habitacional: {s_row['densidad_hab_ha']:.1f} hab/ha ({s_row['densidad_hab_km2']:.1f} hab/km²)")
    print(f"• Manzanas Intersectadas: {len(detalles)} manzanas")
    print(f"• Cobertura de Manzanas dentro del Círculo: {s_row['porcentaje_cobertura_manzanas']:.1f}%")
    print(f"\nDesglose de Manzanas Intersectadas:")
    cols_mostrar = [
        "id_manzana", "n_per_manzana", "area_interseccion_m2",
        "porcentaje_manzana_en_radio", "poblacion_estimada_en_radio", "porcentaje_aporte_al_sensor"
    ]
    display_df = detalles[cols_mostrar].copy()
    display_df.columns = ["ID Manzana", "Pob Manzana", "Área en Círculo (m²)", "% Manzana en Círculo", "Pob en Círculo", "% Aporte Círculo"]
    print(display_df.to_string(index=False))
    
    # Generar visualización local
    idx_sensor = gdf_sensors_3857[gdf_sensors_3857["sensor_id"] == str(sensor_id_consulta)].index[0]
    circulo_sensor = gdf_buffers_3857.loc[idx_sensor].geometry
    
    fig, ax = plt.subplots(figsize=(9, 9), facecolor="#f8fafc")
    ax.set_facecolor("#ffffff")
    
    # Manzanas en el entorno ampliado (buffer de 350m)
    caja_entorno = circulo_sensor.buffer(200.0)
    idxs_entorno = list(sindex_pob.query(caja_entorno, predicate="intersects"))
    if idxs_entorno:
        gdf_pob_3857.iloc[idxs_entorno].plot(
            ax=ax, color="#f1f5f9", edgecolor="#94a3b8", linewidth=1.0, alpha=0.9
        )
    
    # Manzanas intersectadas destacadas
    if not detalles.empty:
        ids_mz = detalles["id_manzana"].astype(str).tolist()
        mzs_int = gdf_pob_3857[gdf_pob_3857["id_manzana"].isin(ids_mz)]
        mzs_int.plot(ax=ax, color="#fed7aa", edgecolor="#ea580c", linewidth=1.5, alpha=0.7)
        
    # Círculo del buffer
    x_c, y_c = circulo_sensor.exterior.xy
    ax.plot(x_c, y_c, color="#2563eb", linewidth=2.5, linestyle="--", label=f"Radio {int(RADIO_SCAN_M)}m")
    
    # Sensor central
    pt = gdf_sensors_3857.loc[idx_sensor].geometry
    ax.scatter([pt.x], [pt.y], color="#dc2626", edgecolor="black", s=130, zorder=6, label=f"Sensor {s_row['sensor_id']}")
    
    # Etiquetas con porcentaje de cada manzana, porcentaje de aporte y personas estimadas
    etiquetas_datos = []
    for _, row_det in detalles.iterrows():
        mz_id_str = str(row_det["id_manzana"])
        geom_m = gdf_pob_3857[gdf_pob_3857["id_manzana"] == mz_id_str]
        if geom_m.empty:
            continue
        geom_mz = geom_m.geometry.iloc[0]
        inter_geom = geom_mz.intersection(circulo_sensor)
        if inter_geom.is_empty:
            continue
        rp = inter_geom.representative_point()
        pct_mz = row_det["porcentaje_manzana_en_radio"]
        pct_pob = row_det["porcentaje_aporte_al_sensor"]
        pob_est = row_det["poblacion_estimada_en_radio"]
        
        if pob_est < 0.05:
            pob_str = "0 hab"
        elif pob_est < 10:
            pob_str = f"{pob_est:.1f} hab"
        else:
            pob_str = f"{pob_est:.0f} hab"
            
        texto = f"%Mz: {pct_mz:.1f}%\n%Pob: {pct_pob:.1f}%\n({pob_str})"
        etiquetas_datos.append({"x": rp.x, "y": rp.y, "text": texto})

    # Algoritmo de relajación para evitar solapamientos entre etiquetas contiguas
    coords = [[it["x"], it["y"]] for it in etiquetas_datos]
    box_w = 52.0
    box_h = 32.0
    for _ in range(40):
        for i in range(len(coords)):
            for j in range(i + 1, len(coords)):
                dx = coords[j][0] - coords[i][0]
                dy = coords[j][1] - coords[i][1]
                if abs(dx) < box_w and abs(dy) < box_h:
                    overlap_x = box_w - abs(dx)
                    overlap_y = box_h - abs(dy)
                    sign_x = 1.0 if dx >= 0 else -1.0
                    sign_y = 1.0 if dy >= 0 else -1.0
                    if overlap_y < overlap_x:
                        shift = overlap_y / 2.0 + 1.0
                        coords[j][1] += sign_y * shift
                        coords[i][1] -= sign_y * shift
                    else:
                        shift = overlap_x / 2.0 + 1.0
                        coords[j][0] += sign_x * shift
                        coords[i][0] -= sign_x * shift

    for k, it in enumerate(etiquetas_datos):
        x_pos, y_pos = coords[k]
        ax.annotate(
            it["text"],
            xy=(x_pos, y_pos),
            fontsize=6.8,
            fontweight="bold",
            color="#7c2d12",
            ha="center",
            va="center",
            bbox=dict(
                boxstyle="round,pad=0.22",
                facecolor="#ffedd5",
                edgecolor="#ea580c",
                alpha=0.92,
                linewidth=0.7
            ),
            zorder=7
        )

    # Encuadre
    c_x, c_y = pt.x, pt.y
    r_view = RADIO_SCAN_M + 120.0
    ax.set_xlim(c_x - r_view, c_x + r_view)
    ax.set_ylim(c_y - r_view, c_y + r_view)
    ax.set_aspect("equal")
    
    # Formato de nombre legible de la ciudad para el título
    MAPA_CIUDADES = {
        "valdivia": "Valdivia",
        "padrelascasas": "Padre Las Casas",
        "temuco": "Temuco",
        "laserena": "La Serena",
        "coquimbo": "Coquimbo",
        "santiago": "Santiago",
    }
    tag_raiz = NOMBRE_CIUDAD.split('_')[0].lower()
    ciudad_nombre = MAPA_CIUDADES.get(tag_raiz, tag_raiz.capitalize())

    ax.set_title(
        f"Entorno del Sensor {s_row['sensor_id']}: {ciudad_nombre} - {s_row['calle_sensor']}\n"
        f"Población en Radio: {s_row['poblacion_total_en_radio']:.1f} hab | {s_row['decibels']:.1f} dB",
        fontsize=11, fontweight="bold", pad=10
    )
    
    custom_legend = [
        Patch(facecolor="#fed7aa", edgecolor="#ea580c", label="Manzanas (% Mz / % Pob aportada / Hab)"),
        Line2D([0], [0], color="#2563eb", linewidth=2.5, linestyle="--", label=f"Radio {int(RADIO_SCAN_M)}m"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="#dc2626", markeredgecolor="black", markersize=10, label=f"Sensor {s_row['sensor_id']}")
    ]
    ax.legend(handles=custom_legend, loc="upper right", frameon=True, facecolor="white", edgecolor="#cbd5e1", fontsize=8.5)
    ax.grid(True, linestyle="--", alpha=0.3, color="#94a3b8")
    
    fig_sensor_out = FIGS_DIR / f"{ciudad_tag}_{s_row['sensor_id']}_poblacion.png"
    plt.savefig(fig_sensor_out, dpi=180, bbox_inches="tight")
    plt.close(fig)
    print(f"\n✔ Figura del entorno guardada en: {fig_sensor_out.name}")

# Ejecutar demostración con el sensor que tiene mayor población
sensor_max_pob = df_sensores_resumen.sort_values(by="poblacion_total_en_radio", ascending=False).iloc[0]["sensor_id"]
inspeccionar_sensor(sensor_max_pob)

FICHA TÉCNICA — SENSOR G-6 (Alcalde Jorge Bustos, nº803)
• Vía OGUC: Local | Nivel de Ruido: 68.6 dB
• Radio de Análisis: 150.0 m | Área del círculo: 7.07 ha
• Población Estimada en Radio: 842.3 habitantes
• Densidad Habitacional: 119.2 hab/ha (11917.0 hab/km²)
• Manzanas Intersectadas: 20 manzanas
• Cobertura de Manzanas dentro del Círculo: 81.7%

Desglose de Manzanas Intersectadas:
    ID Manzana  Pob Manzana  Área en Círculo (m²)  % Manzana en Círculo  Pob en Círculo  % Aporte Círculo
14101081008038         21.0                795.79               16.1844          3.3987            0.4035
14101081008036          0.0               1585.95               83.6386          0.0000            0.0000
14101081008031         19.0               2067.29              100.0000         19.0000            2.2556
14101081008027        310.0               1725.96               14.0938         43.6907            5.1868
14101081008028         27.0               4201.16              100.0000         27.